In [1]:
import joblib
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

In [4]:
test_df = pd.read_csv("C://Users//Diluni_128033//reqsys//data//testData.csv")
best_model = joblib.load('model_svm_tuned.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')

In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [6]:
def clean_requirement(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

In [7]:
print("Processing new client requirements...")
test_df['cleaned'] = test_df['requirement_sentence'].apply(clean_requirement)

Processing new client requirements...


In [8]:
X_test = tfidf.transform(test_df['cleaned'])

In [9]:
predictions = best_model.predict(X_test)

In [10]:
label_names = ['NFR_boolean', 'Security', 'Reliability']
pred_df = pd.DataFrame(predictions, columns=label_names)
results = pd.concat([test_df['requirement_sentence'], pred_df], axis=1)

In [11]:
display(results)

,requirement_sentence,NFR_boolean,Security,Reliability
0,The system shall provide the ability to record...,1,1,0
1,The user ID and date or time stamp shall be re...,1,1,0
2,The system shall provide the ability to cosign...,0,0,0
3,The system shall provide the ability to record...,0,0,0
4,The system shall provide the ability to record...,1,1,0
5,The system shall allow authorized users to upd...,1,1,0
6,The system shall provide the ability to genera...,0,0,0
7,The system shall provide the ability to export...,0,0,0
8,This export on hardcopy and electronic output ...,0,0,0
9,The system shall provide the ability to create...,0,0,0


In [12]:
has_security = results['Security'].any()
has_reliability = results['Reliability'].any()

In [13]:
if not has_security:
    print("ALERT: No Security requirements detected in this upload!")
    print("RECOMMENDATION: Please include requirements regarding data encryption, user authentication, or access logs.")

if not has_reliability:
    print("ALERT: No Reliability requirements detected!")
    print("RECOMMENDATION: Consider adding requirements for system uptime, data backup, or error handling.")

if has_security and has_reliability:
    print("SUCCESS: Requirement set appears comprehensive.")

ALERT: No Reliability requirements detected!
RECOMMENDATION: Consider adding requirements for system uptime, data backup, or error handling.


In [14]:
results.to_csv('validated_requirements.csv', index=False)

print("\nFile saved successfully as 'validated_requirements.csv'")


File saved successfully as 'validated_requirements.csv'
